In [1]:
import time

import cv2
import dxcam
import numpy as np
import torch
import torch.nn.functional as F
from ultralytics import YOLO

from code_programm.path import (get_path_config_road_area_size, get_path_config_speed_area_size)
from code_programm.path import get_path_weight_model


In [2]:
def get_road_area_size():
    with open(get_path_config_road_area_size(), 'r') as file:
        lines = file.readlines()
    return [int(lines[1]), int(lines[1]) + int(lines[3]), int(lines[0]), int(lines[0]) + int(lines[2])]


def get_speed_area_size():
    with open(get_path_config_speed_area_size(), 'r') as file:
        lines = file.readlines()
    return [int(lines[1]), int(lines[1]) + int(lines[3]), int(lines[0]), int(lines[0]) + int(lines[2])]


road_area = get_road_area_size()
speed_area = get_speed_area_size()

In [3]:
try:
    model_road = YOLO(get_path_weight_model('best.pt'))
    model_speed = YOLO(get_path_weight_model('speed_recognition.pt'))
    camera = dxcam.create(device_idx=0, output_color="BGR")
    camera.start(target_fps=60)
except Exception as e:
    print(1)

In [5]:
combined_mask_old = torch.zeros((1, 1, 96, 128), device='cuda')
combined_mask_new = torch.zeros((1, 1, 96, 128), device='cuda')
while True:
    start_time = time.time()
    screenshot_dxcam = camera.get_latest_frame()
    road = screenshot_dxcam[road_area[0]:road_area[1], road_area[2]:road_area[3]]
    speed = screenshot_dxcam[speed_area[0]:speed_area[1], speed_area[2]:speed_area[3]]
    results = model_road.predict(road, conf=0.3, verbose=False, device='cuda', show=True)

    if results[0].masks is not None:
        combined_mask_old = combined_mask_new
        combined_mask_new.zero_()  # Reset the combined mask
        for i in results[0].masks.data:
            resized_mask = F.interpolate(i.unsqueeze(0).unsqueeze(0), size=(96, 128), mode='bilinear', align_corners=False)
            combined_mask_new += resized_mask.squeeze(0).unsqueeze(0)

    combined_tensor = combined_mask_old + combined_mask_new
    combined_mask_cpu = combined_tensor.squeeze(0).squeeze(0).cpu().numpy()
    cv2.imshow('road', combined_mask_cpu)
    print(time.time() - start_time)

0.01902151107788086
0.17456793785095215
0.5730104446411133
0.029394865036010742
0.016001462936401367
0.014542818069458008
0.015722274780273438
0.016998291015625
0.02957320213317871
0.015000343322753906
0.016000747680664062
0.014999151229858398
0.03200268745422363
0.02963852882385254
0.015002250671386719
0.06333589553833008
0.0625617504119873
0.01600050926208496
0.01700115203857422
0.029605627059936523
0.031000614166259766
0.015188932418823242
0.01600050926208496
0.015555143356323242
0.017000436782836914
0.03176593780517578
0.016003847122192383
0.015996217727661133
0.028998613357543945
0.031000137329101562
0.03710031509399414
0.02535700798034668
0.01599907875061035
0.3913748264312744
0.01505422592163086
0.016000747680664062
0.03170585632324219
0.23714709281921387
0.028999805450439453
0.01599884033203125
0.22299718856811523
0.03190445899963379


KeyboardInterrupt: 

In [6]:
cv2.destroyAllWindows()

In [4]:
combined_mask_old = torch.zeros((1, 1, 96, 128), device='cuda')
combined_mask_new = torch.zeros((1, 1, 96, 128), device='cuda')
# try:
while True:
    start_time = time.time()
    screenshot_dxcam = camera.get_latest_frame()
    road = screenshot_dxcam[road_area[0]:road_area[1], road_area[2]:road_area[3]]
    speed = screenshot_dxcam[speed_area[0]:speed_area[1], speed_area[2]:speed_area[3]]
    results = model_road.predict(road, 
                                 conf=0.3, 
                                 verbose=False, 
                                 device='cuda', 
                                 show=True)

    if results[0].masks is not None:
        combined_mask_old = combined_mask_new
        combined_mask_new.zero_()  # Reset the combined mask
        for i in results[0].masks.data:
            resized_mask = F.interpolate(i.unsqueeze(0).unsqueeze(0), size=(96, 128), mode='bilinear', align_corners=False)
            combined_mask_new += resized_mask.squeeze(0).unsqueeze(0)
    
    combined_tensor = combined_mask_old + combined_mask_new
    combined_mask_cpu = combined_tensor.squeeze(0).squeeze(0).cpu().numpy()
    cv2.imshow('road', combined_mask_cpu)
    print(time.time() - start_time)
# except ValueError:
#     cv2.destroyAllWindows()


0.5457630157470703
torch.Size([448, 640])
0.33425164222717285
torch.Size([448, 640])
0.030025720596313477
0.029997587203979492
0.029535770416259766
0.016268014907836914
0.016007423400878906
0.014000415802001953
0.015000104904174805
0.01601409912109375
0.015288829803466797
0.017002105712890625
0.03130006790161133
0.030652523040771484
0.0150909423828125
torch.Size([448, 640])
0.0313875675201416
torch.Size([448, 640])
0.03123950958251953
0.031000614166259766
0.01600956916809082
0.016571044921875
torch.Size([448, 640])
0.029417753219604492
torch.Size([448, 640])
0.03023838996887207
0.015799760818481445
0.015477418899536133
0.015857458114624023
0.01617574691772461
0.015636444091796875
torch.Size([448, 640])
0.03160381317138672
torch.Size([448, 640])
0.03300189971923828
0.014689445495605469
torch.Size([448, 640])
0.03202104568481445
0.014624595642089844
torch.Size([448, 640])
0.032012939453125
0.016176939010620117
0.014543294906616211
torch.Size([448, 640])
0.032537221908569336
torch.Size([4

KeyboardInterrupt: 